# Step 4: Oriented Bounding Box (OBB) Detection

Trains YOLO26-OBB and YOLO11-OBB variants on the OBB dataset derived from the polygon labels (produced by Step 2's polygon→OBB conversion cell). Compares OBB localization to the HBB results from Step 2.

**Brief requirement:** train both YOLO26-OBB and YOLO11-OBB, compare HBB vs OBB, discuss localization precision and failure cases on rotated pool geometries.

**Runtime:** Colab → Runtime → Change runtime type → **A100 GPU** (40 GB VRAM target). T4 / L4 also work with VARIANT_BATCH intact.

**Input:** `pool_dataset_obb.zip` on Drive (produced by `02_train_yolo26.ipynb` Step 9).


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU, switch runtime to GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB  |  torch {torch.__version__}')

In [ ]:
%pip install -q 'ultralytics>=8.4.0' supervision pandas matplotlib opencv-python pyyaml
!yolo settings sync=False
import ultralytics; ultralytics.checks()

## Load OBB dataset from Drive

Expects `pool_dataset_obb.zip` at `MyDrive/IE/IndividualAssignmentMBD2026/`. Produced by the polygon→OBB cell in `02_train_yolo26.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib, shutil
ZIP_PATH = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/pool_dataset_obb.zip'
assert pathlib.Path(ZIP_PATH).exists(), f'Drive input missing at {ZIP_PATH}. Confirm Step 2 finished (it produces pool_dataset_obb.zip in Drive) before running Step 4.'
DATA_DIR = pathlib.Path('/content/pool_dataset_obb')
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)

import yaml
yml = yaml.safe_load(open(DATA_DIR / 'data.yaml'))
yml['train'] = str(DATA_DIR / 'train' / 'images')
yml['val']   = str(DATA_DIR / 'valid' / 'images')
yml['test']  = str(DATA_DIR / 'test'  / 'images')
yaml.safe_dump(yml, open(DATA_DIR / 'data.yaml', 'w'))
DATA_YAML = str(DATA_DIR / 'data.yaml')
print('OBB Data YAML:', DATA_YAML)
print(open(DATA_YAML).read())

# Sanity: count label tokens (OBB format = 9 tokens: class + 4 corners × 2 coords)
import collections
token_counts = collections.Counter()
for split in ['train', 'valid', 'test']:
    for lbl in (DATA_DIR / split / 'labels').iterdir():
        for line in open(lbl):
            n = len(line.split())
            token_counts[n] += 1
print('\nOBB label token-count distribution (expect 9):')
for k, v in sorted(token_counts.items()):
    print(f'  {k} tokens: {v} lines')

## Training configuration

Same hyperparameters as the YOLO26 HBB training for a fair comparison. The OBB head is built into the model architecture; everything else is unchanged.

| Parameter | Value | Notes |
|---|---|---|
| Optimizer | `AdamW` (explicit) | Identical to Step 2; set explicitly so `lr0`/`momentum` actually take effect (`optimizer='auto'` silently overrides them). |
| Initial LR | 0.005 | Identical to Step 2 |
| LR scheduler | Cosine | Identical to Step 2 |
| Epochs | 100, patience=30 | Identical to Step 2 |
| Image size | 640 | Identical to Step 2 |
| Batch size | n=16, s=8 | OBB heads add ~10% memory overhead, slightly smaller batches than HBB |
| Augmentations | Same aerial-tuned set | mosaic=1.0, mixup=0.05, flipud=0.5, degrees=180, etc. |
| `close_mosaic` | 10 | Same |

In [ ]:
BASE_CFG = dict(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,
    optimizer='AdamW',           # explicit (was 'auto'): 'auto' silently overrides lr0/momentum, breaking the hparam-table contract
    lr0=0.005,
    cos_lr=True,
    close_mosaic=10,
    seed=0,
    deterministic=True,
    cache='ram',
    plots=True,
    verbose=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    degrees=180,
    mosaic=1.0, mixup=0.05,
    amp=True,  # explicit AMP for A100 (Ultralytics default is also True; documenting in hparam table)
    exist_ok=True,  # overwrite same-named runs/<name>/ on re-entry; prevents auto-incrementing dir names
)
# Slightly smaller batches than HBB since OBB heads add ~10% memory overhead
VARIANT_BATCH = {
    'yolo26n-obb': 16, 'yolo26s-obb': 8,
    'yolo11n-obb': 16, 'yolo11s-obb': 8,
}
print('BASE_CFG:')
for k, v in BASE_CFG.items(): print(f'  {k:14s} = {v}')
print('\nPer-variant batch:')
for k, v in VARIANT_BATCH.items(): print(f'  {k}: batch={v}')

## Training utility

In [ ]:
from ultralytics import YOLO
import time, gc

results_summary = {}

def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print(f'  GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated / {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

def train_obb(variant: str, cfg=None):
    """Train one OBB variant; return val metrics dict, or None if weights aren't available."""
    if cfg is None:
        cfg = dict(BASE_CFG)
        cfg['batch'] = VARIANT_BATCH.get(variant, 16)
    print(f'\n{"="*60}\nTraining {variant} (batch={cfg["batch"]})\n{"="*60}')
    _free_gpu()
    try:
        model = YOLO(f'{variant}.pt')
    except Exception as e:
        # YOLO26-OBB weights may not be in the installed ultralytics yet.
        # Skip rather than fall back to YOLO11, since YOLO11 variants are trained separately below.
        print(f'  Could not load {variant}.pt ({e.__class__.__name__}); skipping. YOLO11 variants will still run.')
        return None
    try:
        n_params = sum(p.numel() for p in model.model.parameters())
        t0 = time.time()
        train_res = model.train(name=variant, **cfg)
        train_time = time.time() - t0
        val = model.val(data=DATA_YAML, split='val', verbose=False)
    except Exception as e:
        # Train/val failures (OOM, dataset glitch, OBB-head bug in a specific ultralytics build, etc.)
        # must not kill the variant loop; downstream cells iterate results_summary and will skip this variant.
        print(f'  Train/val FAILED for {variant}: {e.__class__.__name__}: {e}; skipping. Subsequent variants will still run.')
        del model
        _free_gpu()
        return None
    record = dict(
        mAP50    = float(val.box.map50),
        mAP50_95 = float(val.box.map),
        precision= float(val.box.mp),
        recall   = float(val.box.mr),
        params   = n_params,
        train_time_s = train_time,
        best_weights = str(train_res.save_dir / 'weights' / 'best.pt'),
    )
    results_summary[variant] = record
    print(f'\n{variant}: mAP50={val.box.map50:.4f}  mAP50-95={val.box.map:.4f}  P={val.box.mp:.4f}  R={val.box.mr:.4f}  params={n_params/1e6:.2f}M  time={train_time/60:.1f}min')
    del model, train_res, val
    _free_gpu()
    return record

## Train YOLO26-OBB (n + s)

Trains the YOLO26 OBB variants. If YOLO26-OBB weights aren't available in your Ultralytics version, `train_obb` prints a skip notice and returns `None`. The YOLO11-OBB cells below run regardless, so you still get a complete OBB comparison.

In [ ]:
for variant in ['yolo26n-obb', 'yolo26s-obb']:
    train_obb(variant)

## Train YOLO11-OBB (n + s)

Brief explicitly requires YOLO11-OBB alongside YOLO26-OBB.

In [ ]:
for variant in ['yolo11n-obb', 'yolo11s-obb']:
    train_obb(variant)

## OBB comparison table

In [ ]:
import pandas as pd
if not results_summary:
    raise RuntimeError(
        'No OBB variants trained successfully. Check the training-loop output above for model-load failures, OOM, or install issues. '
        'All cells below this one depend on results_summary; aborting here so they do not crash with cryptic KeyError/idxmax errors.'
    )
df = pd.DataFrame(results_summary).T
df['params_M'] = df['params'] / 1e6
df = df[['mAP50', 'mAP50_95', 'precision', 'recall', 'params_M', 'train_time_s']]
df.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Params (M)', 'Train time (s)']
df = df.round(4)
print(df.to_string())
df.to_csv('/content/obb_comparison.csv')
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(df.index, df['mAP@50-95'], color='#E84C4C')
axes[0].set_ylabel('mAP@50-95'); axes[0].set_title('OBB validation mAP@50-95')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20)
for i, v in enumerate(df['mAP@50-95']):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9)
axes[1].scatter(df['Params (M)'], df['mAP@50-95'], s=120, c='#E84C4C')
for name, row in df.iterrows():
    axes[1].annotate(name, (row['Params (M)'], row['mAP@50-95']), xytext=(5, 5), textcoords='offset points', fontsize=8)
axes[1].set_xlabel('Parameters (M)'); axes[1].set_ylabel('mAP@50-95'); axes[1].set_title('OBB Accuracy vs Model Size')
plt.tight_layout(); plt.show()

## HBB vs OBB comparison

Pulls the YOLO26 HBB results from Step 2's CSV and puts them side-by-side with the OBB results. The key comparison is **mAP@50-95**, which is sensitive to box-quality (tight fit). OBB should win on rotated/elongated pools.

In [ ]:
import pathlib
hbb_csv = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/yolo26_comparison.csv'
hbb_df = None
if pathlib.Path(hbb_csv).exists():
    hbb_df = pd.read_csv(hbb_csv, index_col=0)
    print('HBB results from Step 2:')
    print(hbb_df.to_string())

obb_df = df  # from previous cell
print('\nOBB results from Step 4:')
print(obb_df.to_string())

if hbb_df is not None:
    cmp_rows = []
    for model_family in ['n', 's']:
        hbb_key = f'yolo26{model_family}'
        obb_key = f'yolo26{model_family}-obb'
        if hbb_key in hbb_df.index and obb_key in obb_df.index:
            cmp_rows.append({
                'variant': model_family,
                'HBB mAP@50':       hbb_df.loc[hbb_key, 'mAP@50'],
                'OBB mAP@50':       obb_df.loc[obb_key, 'mAP@50'],
                'HBB mAP@50-95':    hbb_df.loc[hbb_key, 'mAP@50-95'],
                'OBB mAP@50-95':    obb_df.loc[obb_key, 'mAP@50-95'],
                'HBB Params (M)':   hbb_df.loc[hbb_key, 'Params (M)'],
                'OBB Params (M)':   obb_df.loc[obb_key, 'Params (M)'],
            })
    if cmp_rows:
        cmp_df = pd.DataFrame(cmp_rows).set_index('variant').round(4)
        print('\nSide-by-side comparison (HBB vs OBB):')
        print(cmp_df.to_string())
        cmp_df.to_csv('/content/hbb_vs_obb.csv')
        fig, ax = plt.subplots(figsize=(9, 5))
        x = np.arange(len(cmp_df)); width = 0.2
        ax.bar(x - 1.5*width, cmp_df['HBB mAP@50'],    width, label='HBB mAP@50', color='#4C86E8')
        ax.bar(x - 0.5*width, cmp_df['OBB mAP@50'],    width, label='OBB mAP@50', color='#82B4FF')
        ax.bar(x + 0.5*width, cmp_df['HBB mAP@50-95'], width, label='HBB mAP@50-95', color='#E84C4C')
        ax.bar(x + 1.5*width, cmp_df['OBB mAP@50-95'], width, label='OBB mAP@50-95', color='#FF8282')
        ax.set_xticks(x); ax.set_xticklabels([f'yolo26{v}' for v in cmp_df.index])
        ax.set_ylabel('mAP'); ax.set_title('HBB vs OBB')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout(); plt.show()
else:
    print(f'\nHBB CSV not found at {hbb_csv}, skipping side-by-side comparison')

## Test-set predictions visualised (OBB)

Run the best OBB model on test images and show the rotated boxes.

In [ ]:
import random
from PIL import Image

BEST_OBB = df['mAP@50-95'].idxmax()
print(f'Best OBB variant: {BEST_OBB}')
best_obb_weights = results_summary[BEST_OBB]['best_weights']
best_obb = YOLO(best_obb_weights)

test_imgs = sorted((DATA_DIR / 'test' / 'images').iterdir())
random.seed(0)
sample = random.sample(test_imgs, k=min(9, len(test_imgs)))
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, p in zip(axes.flat, sample):
    img = np.array(Image.open(p))
    ax.imshow(img)
    res = best_obb.predict(str(p), verbose=False, conf=0.25)[0]
    # OBB results have res.obb instead of res.boxes for rotated detections
    obb = getattr(res, 'obb', None)
    if obb is not None and len(obb.xyxyxyxy) > 0:
        for corners, conf in zip(obb.xyxyxyxy.cpu().numpy(), obb.conf.cpu().numpy()):
            pts = np.vstack([corners, corners[0]])
            ax.plot(pts[:, 0], pts[:, 1], 'lime', linewidth=2)
            ax.text(pts[0, 0], pts[0, 1] - 5, f'{conf:.2f}', color='lime', fontsize=8)
    ax.set_title(p.name, fontsize=8); ax.axis('off')
plt.suptitle(f'Test predictions ({BEST_OBB})')
plt.tight_layout(); plt.show()

## Test-set evaluation

In [ ]:
# Test-evaluate every trained variant (parallel to Step 2's yolo26_test_metrics.csv and Step 3's rfdetr_test_metrics.csv)
obb_test_records = {}
for variant, rec in results_summary.items():
    print(f'\nTest-evaluating {variant}')
    model = YOLO(rec['best_weights'])
    val = model.val(data=DATA_YAML, split='test', verbose=False)
    obb_test_records[variant] = dict(
        mAP50     = float(val.box.map50),
        mAP50_95  = float(val.box.map),
        precision = float(val.box.mp),
        recall    = float(val.box.mr),
    )
    print(f'  mAP50={val.box.map50:.4f}  mAP50-95={val.box.map:.4f}  P={val.box.mp:.4f}  R={val.box.mr:.4f}')
    del model, val
    _free_gpu()

obb_test_df = pd.DataFrame(obb_test_records).T.round(4)
obb_test_df.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']
obb_test_df.to_csv('/content/obb_test_metrics.csv')
print('\nFull OBB test metrics:')
print(obb_test_df.to_string())
print(f'\nBest by test mAP@50-95: {obb_test_df["mAP@50-95"].idxmax()}')

## Failure case analysis (test set, best OBB variant)

Parallel to Step 2's `yolo26_failures.zip` and Step 3's `rfdetr_failures.zip`. For each test image we run the best OBB variant (highest val mAP@50-95) at conf=0.25 and classify it as TP-only / FP-only / FN-only / mixed by **rotated** IoU match (via `cv2.rotatedRectangleIntersection`). Annotated images are saved (green = TP, red = FP, yellow = missed GT, all drawn as rotated polygons) and zipped as `obb_failures.zip`. Gives the OBB section the same qualitative substrate as the other steps.

In [ ]:
import cv2, zipfile
from tqdm import tqdm

# BEST_OBB / best_obb / best_obb_weights are defined in the test-viz cell above (by val mAP@50-95)
print(f'Saving OBB failure cases for {BEST_OBB}')

def _obb_iou(p1, p2):
    """IoU of two OBBs given as 4x2 corner arrays in pixel coords (uses cv2 rotated-rect intersection)."""
    rect1 = cv2.minAreaRect(p1.astype(np.float32))
    rect2 = cv2.minAreaRect(p2.astype(np.float32))
    ret, inter_pts = cv2.rotatedRectangleIntersection(rect1, rect2)
    if ret == 0 or inter_pts is None or len(inter_pts) < 3:
        return 0.0
    inter = cv2.contourArea(cv2.convexHull(inter_pts.astype(np.float32)))
    a1 = cv2.contourArea(p1.astype(np.float32))
    a2 = cv2.contourArea(p2.astype(np.float32))
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

def _load_gt_obbs(lbl_path, W, H):
    """Read a YOLO OBB label file, return list of 4x2 corner arrays in pixel coords."""
    boxes = []
    if not lbl_path.exists(): return boxes
    for line in open(lbl_path):
        parts = line.split()
        if len(parts) < 9: continue
        coords = np.array(list(map(float, parts[1:9]))).reshape(4, 2)
        coords[:, 0] *= W; coords[:, 1] *= H
        boxes.append(coords)
    return boxes

failures_dir = pathlib.Path('/content/obb_failures')
if failures_dir.exists(): shutil.rmtree(failures_dir)
failures_dir.mkdir()

CONF_THR, IOU_THR = 0.25, 0.5
counts = {'tp_only_or_clean': 0, 'fp_only': 0, 'fn_only': 0, 'mixed': 0}

test_imgs = sorted((DATA_DIR / 'test' / 'images').iterdir())
for p in tqdm(test_imgs, desc='obb failures'):
    img = np.array(Image.open(p).convert('RGB'))
    H, W = img.shape[:2]
    res = best_obb.predict(str(p), verbose=False, conf=CONF_THR)[0]
    obb_pred = getattr(res, 'obb', None)
    if obb_pred is not None and len(obb_pred.xyxyxyxy) > 0:
        det_corners = obb_pred.xyxyxyxy.cpu().numpy()
        det_conf    = obb_pred.conf.cpu().numpy()
    else:
        det_corners = np.empty((0, 4, 2))
        det_conf    = np.empty(0)

    gt_corners = _load_gt_obbs(DATA_DIR / 'test' / 'labels' / (p.stem + '.txt'), W, H)

    matched_gt, tp_idx, fp_idx = set(), [], []
    if len(det_corners) and len(gt_corners):
        for di in np.argsort(-det_conf):
            best_gt, best_iou = -1, IOU_THR
            for gj in range(len(gt_corners)):
                if gj in matched_gt: continue
                iou = _obb_iou(det_corners[di], gt_corners[gj])
                if iou >= best_iou:
                    best_iou, best_gt = iou, gj
            if best_gt >= 0:
                matched_gt.add(best_gt); tp_idx.append(di)
            else:
                fp_idx.append(di)
    else:
        fp_idx = list(range(len(det_corners)))
    fn_idx = [gj for gj in range(len(gt_corners)) if gj not in matched_gt]

    n_fp, n_fn = len(fp_idx), len(fn_idx)
    if n_fp == 0 and n_fn == 0:
        counts['tp_only_or_clean'] += 1
        continue
    category = 'mixed' if (n_fp > 0 and n_fn > 0) else ('fp_only' if n_fp > 0 else 'fn_only')
    counts[category] += 1

    # Color tuples are in RGB order (NOT cv2's usual BGR), because `img` came from PIL → RGB
    # and we save via PIL → RGB. cv2 draw funcs just write the tuple as raw channel bytes.
    arr = img.copy()
    def _draw(poly, color, label):
        pts = poly.astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(arr, [pts], isClosed=True, color=color, thickness=2)
        cv2.putText(arr, label, tuple(poly[0].astype(int) + np.array([0, -5])), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    for i in tp_idx:
        _draw(det_corners[i], (0, 255, 0), f'TP {det_conf[i]:.2f}')       # green
    for i in fp_idx:
        _draw(det_corners[i], (255, 0, 0), f'FP {det_conf[i]:.2f}')       # red
    for j in fn_idx:
        _draw(gt_corners[j], (255, 255, 0), 'FN')                          # yellow (R+G)

    Image.fromarray(arr).save(failures_dir / f'{category}_{p.stem}.png')

print(f'\nFailure breakdown for {BEST_OBB} on test:')
for cat, n in counts.items():
    print(f'  {cat}: {n}')

zip_path = '/content/obb_failures.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in failures_dir.iterdir():
        zf.write(f, f.name)
print(f'\nSaved {zip_path} ({sum(1 for _ in failures_dir.iterdir())} annotated images)')

## Localization precision: HBB vs OBB on same test image

Picks a few test images with elongated pools (high aspect ratio) and renders both the HBB prediction (axis-aligned from Step 2 best model) and the OBB prediction side-by-side. This is the figure to put in the "When are OBB annotations beneficial?" section of the write-up.

In [ ]:
from matplotlib.patches import Rectangle

# Load Step 2 best HBB model (from Drive zip extraction if needed)
hbb_runs_zip = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/yolo26_runs.zip'
hbb_runs_dir = pathlib.Path('/content/hbb_runs')
if hbb_runs_dir.exists(): shutil.rmtree(hbb_runs_dir)
hbb_runs_dir.mkdir()
if pathlib.Path(hbb_runs_zip).exists():
    with zipfile.ZipFile(hbb_runs_zip) as z: z.extractall(hbb_runs_dir)
    print('Extracted Step 2 HBB runs')
    hbb_csv_path = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/yolo26_comparison.csv'
    if pathlib.Path(hbb_csv_path).exists():
        _hbb_df = pd.read_csv(hbb_csv_path, index_col=0)
        best_hbb_name = _hbb_df['mAP@50-95'].idxmax()
        candidates = list(hbb_runs_dir.glob(f'detect/{best_hbb_name}/weights/best.pt'))
        if not candidates:
            candidates = list(hbb_runs_dir.glob(f'**/{best_hbb_name}/weights/best.pt'))
        if candidates:
            hbb_model = YOLO(str(candidates[0]))
            print(f'Loaded HBB model: {best_hbb_name} from {candidates[0]}')

            # Find test images with elongated pools (aspect ratio > 1.5) using the OBB labels
            elongated = []
            for img_path in sorted((DATA_DIR / 'test' / 'images').iterdir()):
                lbl_path = DATA_DIR / 'test' / 'labels' / (img_path.stem + '.txt')
                if not lbl_path.exists(): continue
                for line in open(lbl_path):
                    parts = line.split()
                    if len(parts) < 9: continue
                    coords = np.array(list(map(float, parts[1:]))).reshape(-1, 2)
                    sides = [np.linalg.norm(coords[(i+1)%4] - coords[i]) for i in range(4)]
                    aspect = max(sides) / max(min(sides), 1e-6)
                    if aspect > 1.5:
                        elongated.append((img_path, aspect))
                        break
            elongated.sort(key=lambda x: -x[1])
            sample = elongated[:6]
            print(f'Found {len(elongated)} test images with elongated pools, showing top 6')

            if not sample:
                # No elongated pools => skip the side-by-side figure rather than crash on plt.subplots(0, 2, ...).
                # Fall back to a low aspect-ratio threshold so the localization figure is still informative.
                print('No pools with aspect>1.5 in test set; rebuilding sample at aspect>1.2 as a fallback.')
                elongated = []
                for img_path in sorted((DATA_DIR / 'test' / 'images').iterdir()):
                    lbl_path = DATA_DIR / 'test' / 'labels' / (img_path.stem + '.txt')
                    if not lbl_path.exists(): continue
                    for line in open(lbl_path):
                        parts = line.split()
                        if len(parts) < 9: continue
                        coords = np.array(list(map(float, parts[1:]))).reshape(-1, 2)
                        sides = [np.linalg.norm(coords[(i+1)%4] - coords[i]) for i in range(4)]
                        aspect = max(sides) / max(min(sides), 1e-6)
                        if aspect > 1.2:
                            elongated.append((img_path, aspect))
                            break
                elongated.sort(key=lambda x: -x[1])
                sample = elongated[:6]
                print(f'Fallback found {len(elongated)} images with aspect>1.2')

            if not sample:
                print('Still no elongated pools available — skipping HBB-vs-OBB localization figure.')
            else:
                fig, axes = plt.subplots(len(sample), 2, figsize=(12, 4*len(sample)))
                if len(sample) == 1: axes = axes.reshape(1, -1)
                for row, (p, aspect) in enumerate(sample):
                    img = np.array(Image.open(p))
                    # HBB
                    hbb_res = hbb_model.predict(str(p), verbose=False, conf=0.25)[0]
                    axes[row, 0].imshow(img)
                    for box in hbb_res.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        conf = float(box.conf[0])
                        axes[row, 0].add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='cyan', linewidth=2))
                        axes[row, 0].text(x1, y1-5, f'{conf:.2f}', color='cyan', fontsize=9)
                    axes[row, 0].set_title(f'HBB aspect={aspect:.1f}\n{p.name[:25]}', fontsize=9); axes[row, 0].axis('off')
                    # OBB
                    obb_res = best_obb.predict(str(p), verbose=False, conf=0.25)[0]
                    axes[row, 1].imshow(img)
                    obb_pred = getattr(obb_res, 'obb', None)
                    if obb_pred is not None and len(obb_pred.xyxyxyxy) > 0:
                        for corners, conf in zip(obb_pred.xyxyxyxy.cpu().numpy(), obb_pred.conf.cpu().numpy()):
                            pts = np.vstack([corners, corners[0]])
                            axes[row, 1].plot(pts[:, 0], pts[:, 1], 'lime', linewidth=2)
                            axes[row, 1].text(pts[0, 0], pts[0, 1]-5, f'{conf:.2f}', color='lime', fontsize=9)
                    axes[row, 1].set_title(f'OBB (same image)\n{best_obb_weights.split("/")[-3]}', fontsize=9); axes[row, 1].axis('off')
                plt.tight_layout(); plt.show()
            del hbb_model; _free_gpu()
        else:
            print(f'Could not find best.pt for {best_hbb_name}, comparison plot skipped')
    else:
        print('HBB comparison CSV missing, skipping')
else:
    print(f'{hbb_runs_zip} not found, skipping HBB comparison; train Step 2 first')

## Bundle artifacts to Drive

In [ ]:
DRIVE_OUT = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026'

# Training runs (all variants' weights, curves, val previews)
if pathlib.Path('/content/runs').exists():
    shutil.make_archive('/content/obb_runs', 'zip', '/content/runs')
    shutil.copy('/content/obb_runs.zip', f'{DRIVE_OUT}/obb_runs.zip')
    _runs_ok = True
else:
    _runs_ok = False

# CSVs and failures zip
artifacts = [
    ('obb_comparison.csv',   'OBB val variant comparison'),
    ('obb_test_metrics.csv', 'OBB test metrics (all variants)'),
    ('hbb_vs_obb.csv',       'HBB vs OBB side-by-side (if Step 2 results present)'),
    ('obb_failures.zip',     'best OBB variant failure cases (TP/FP/FN annotated)'),
]
print('Saved to Drive:')
if _runs_ok:
    print(f'  obb_runs.zip          : all OBB training outputs (curves, weights, val previews)')
else:
    print(f'  obb_runs.zip          : SKIPPED (no /content/runs — no training succeeded)')
for fname, desc in artifacts:
    src = f'/content/{fname}'
    if pathlib.Path(src).exists():
        shutil.copy(src, f'{DRIVE_OUT}/{fname}')
        print(f'  {fname:22s}: {desc}')
    else:
        print(f'  {fname:22s}: SKIPPED (not present)')

## Hardware used (for the write-up)

Auto-detected in the first cell. Same hardware as Step 2 for fair comparison.